In [1]:
# ============================================================
# GEMMA 4 E2B — OPENAI-COMPATIBLE API
# ============================================================

!pip install -q fastapi uvicorn requests pillow openai

import os
import re
import time
import base64
import threading
import uuid
import io
from typing import Any

from openai import OpenAI

import requests
import torch

from PIL import Image
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel, Field
import uvicorn


# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_ID = "google/gemma-4-e2b-it"

HOST = "0.0.0.0"
PORT = 8000

# Generation limits
DEFAULT_MAX_TOKENS = 512
MAX_MAX_TOKENS = 4096

DEFAULT_TEMPERATURE = 0.2
DEFAULT_TOP_P = 0.95

# Public API authentication
API_KEY = os.environ.get("GEMMA_API_KEY")

if not API_KEY:
    API_KEY = (
        "kaggle-"
        + base64.urlsafe_b64encode(os.urandom(24))
        .decode()
        .rstrip("=")
    )

    os.environ["GEMMA_API_KEY"] = API_KEY

    print("Generated a new GEMMA_API_KEY.")
    print("It is stored in the environment only.")
else:
    print("Using existing GEMMA_API_KEY from environment.")


# ============================================================
# 2. FASTAPI APP
# ============================================================

app = FastAPI(
    title="Gemma 4 E2B OpenAI-Compatible API",
    version="1.0.0",
)


# One GPU model -> serialize generation.
generation_lock = threading.Lock()


# ============================================================
# 3. REQUEST SCHEMA
# ============================================================

class ChatCompletionRequest(BaseModel):

    model: str | None = None

    messages: list[dict[str, Any]]

    max_tokens: int | None = Field(
        default=DEFAULT_MAX_TOKENS,
        ge=1,
        le=MAX_MAX_TOKENS,
    )

    temperature: float | None = Field(
        default=DEFAULT_TEMPERATURE,
        ge=0.0,
        le=2.0,
    )

    top_p: float | None = Field(
        default=DEFAULT_TOP_P,
        gt=0.0,
        le=1.0,
    )

    reasoning_effort: str | None = None

    stream: bool | None = False


# ============================================================
# 4. AUTHENTICATION
# ============================================================

def check_auth(authorization: str | None):

    expected = f"Bearer {API_KEY}"

    if authorization != expected:
        raise HTTPException(
            status_code=401,
            detail="Invalid API key",
        )


# ============================================================
# 5. IMAGE LOADING
# ============================================================

def load_image_from_url(url: str):

    if url.startswith("data:image/"):

        try:
            _, encoded = url.split(",", 1)

            raw = base64.b64decode(encoded)

            image = Image.open(
                io.BytesIO(raw)
            ).convert("RGB")

            return image

        except Exception as e:

            raise HTTPException(
                status_code=400,
                detail=f"Invalid base64 image: {e}",
            )

    if url.startswith(("http://", "https://")):

        try:

            response = requests.get(
                url,
                timeout=30,
            )

            response.raise_for_status()

            image = Image.open(
                io.BytesIO(response.content)
            ).convert("RGB")

            return image

        except Exception as e:

            raise HTTPException(
                status_code=400,
                detail=f"Could not download image: {e}",
            )

    raise HTTPException(
        status_code=400,
        detail=(
            "Only HTTP(S) image URLs and "
            "image data URLs are supported."
        ),
    )


# ============================================================
# 6. NORMALIZE OPENAI MESSAGES
# ============================================================

def normalize_messages(messages):

    normalized = []

    for msg in messages:

        role = msg.get("role", "user")

        content = msg.get(
            "content",
            "",
        )

        # ----------------------------------------------------
        # Plain text
        # ----------------------------------------------------

        if isinstance(content, str):

            normalized.append(
                {
                    "role": role,
                    "content": [
                        {
                            "type": "text",
                            "text": content,
                        }
                    ],
                }
            )

            continue

        # ----------------------------------------------------
        # Multimodal
        # ----------------------------------------------------

        if not isinstance(content, list):

            raise HTTPException(
                status_code=400,
                detail=(
                    "message.content must be "
                    "a string or a list."
                ),
            )

        parts = []

        for item in content:

            item_type = item.get("type")

            # ------------------------------------------------
            # Text
            # ------------------------------------------------

            if item_type == "text":

                parts.append(
                    {
                        "type": "text",
                        "text": item.get(
                            "text",
                            "",
                        ),
                    }
                )

            # ------------------------------------------------
            # OpenAI image_url
            # ------------------------------------------------

            elif item_type == "image_url":

                image_url = item.get(
                    "image_url",
                    {}
                )

                if isinstance(image_url, dict):

                    url = image_url.get("url")

                else:

                    url = image_url

                if not url:

                    raise HTTPException(
                        status_code=400,
                        detail=(
                            "image_url.url is missing."
                        ),
                    )

                image = load_image_from_url(url)

                parts.append(
                    {
                        "type": "image",
                        "image": image,
                    }
                )

            # ------------------------------------------------
            # Native HF-style image
            # ------------------------------------------------

            elif item_type == "image":

                if "image" in item:

                    image_value = item["image"]

                    if isinstance(
                        image_value,
                        Image.Image,
                    ):
                        image = image_value

                    elif isinstance(
                        image_value,
                        str,
                    ):
                        image = load_image_from_url(
                            image_value
                        )

                    else:

                        raise HTTPException(
                            status_code=400,
                            detail=(
                                "Unsupported image value."
                            ),
                        )

                elif "url" in item:

                    image = load_image_from_url(
                        item["url"]
                    )

                else:

                    raise HTTPException(
                        status_code=400,
                        detail=(
                            "Image item requires "
                            "'image' or 'url'."
                        ),
                    )

                parts.append(
                    {
                        "type": "image",
                        "image": image,
                    }
                )

            else:

                raise HTTPException(
                    status_code=400,
                    detail=(
                        f"Unsupported content type: "
                        f"{item_type}"
                    ),
                )

        normalized.append(
            {
                "role": role,
                "content": parts,
            }
        )

    return normalized


# ============================================================
# 7. EXTRACT GENERATED TEXT
# ============================================================

def clean_generated_text(text: str):

    if not text:
        return ""

    # Remove Gemma turn markers that occasionally appear
    text = text.replace("<turn|>", "")

    # Remove common trailing generation markers
    text = text.replace("<end_of_turn>", "")

    return text.strip()


def extract_text(output):

    # --------------------------------------------------------
    # Pipeline output
    # --------------------------------------------------------

    if isinstance(output, list) and output:

        generated = output[0].get(
            "generated_text",
            output[0],
        )

    else:

        generated = output

    # --------------------------------------------------------
    # Plain string
    # --------------------------------------------------------

    if isinstance(generated, str):

        return clean_generated_text(
            generated
        )

    # --------------------------------------------------------
    # Chat-style output
    # --------------------------------------------------------

    if isinstance(generated, list):

        # Prefer assistant message
        for item in reversed(generated):

            if not isinstance(item, dict):
                continue

            if item.get("role") != "assistant":
                continue

            content = item.get(
                "content",
                "",
            )

            if isinstance(content, str):

                return clean_generated_text(
                    content
                )

            if isinstance(content, list):

                texts = []

                for part in content:

                    if (
                        isinstance(part, dict)
                        and part.get("type") == "text"
                    ):
                        texts.append(
                            part.get(
                                "text",
                                "",
                            )
                        )

                return clean_generated_text(
                    "".join(texts)
                )

        # Fallback
        texts = []

        for item in generated:

            if not isinstance(item, dict):
                continue

            if isinstance(
                item.get("text"),
                str,
            ):
                texts.append(
                    item["text"]
                )

        return clean_generated_text(
            "".join(texts)
        )

    return clean_generated_text(
        str(generated)
    )


# ============================================================
# 8. GENERATION CONFIG
# ============================================================

def build_generation_kwargs(request):

    max_tokens = request.max_tokens

    if max_tokens is None:
        max_tokens = DEFAULT_MAX_TOKENS

    max_tokens = max(
        1,
        min(
            max_tokens,
            MAX_MAX_TOKENS,
        ),
    )

    kwargs = {
        "max_new_tokens": max_tokens,
    }

    # --------------------------------------------------------
    # Sampling
    # --------------------------------------------------------

    temperature = request.temperature

    if temperature is not None:

        if temperature > 0:

            kwargs["do_sample"] = True
            kwargs["temperature"] = temperature

        else:

            kwargs["do_sample"] = False

    top_p = request.top_p

    if (
        top_p is not None
        and temperature is not None
        and temperature > 0
    ):

        kwargs["top_p"] = top_p

    return kwargs


# ============================================================
# 9. HEALTH
# ============================================================

@app.get("/health")
def health():

    return {
        "status": "ok",
        "model": MODEL_ID,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
    }


# ============================================================
# 10. MODELS
# ============================================================

@app.get("/v1/models")
def models(
    authorization: str | None = Header(
        default=None
    )
):

    check_auth(authorization)

    return {
        "object": "list",
        "data": [
            {
                "id": MODEL_ID,
                "object": "model",
                "created": int(time.time()),
                "owned_by": "google",
            }
        ],
    }


# ============================================================
# 11. CHAT COMPLETIONS
# ============================================================

@app.post("/v1/chat/completions")
def chat_completions(
    request: ChatCompletionRequest,
    authorization: str | None = Header(
        default=None
    ),
):

    check_auth(authorization)

    # --------------------------------------------------------
    # Model validation
    # --------------------------------------------------------

    accepted_models = {
        MODEL_ID,
        "gemma-4-e2b-it",
        "gemma-4-e2b",
    }

    if (
        request.model
        and request.model not in accepted_models
    ):

        raise HTTPException(
            status_code=404,
            detail=(
                f"Unknown model: "
                f"{request.model}"
            ),
        )

    # --------------------------------------------------------
    # Streaming
    # --------------------------------------------------------

    if request.stream:

        raise HTTPException(
            status_code=400,
            detail=(
                "stream=true is not implemented."
            ),
        )

    # --------------------------------------------------------
    # Normalize input
    # --------------------------------------------------------

    messages = normalize_messages(
        request.messages
    )

    generation_kwargs = build_generation_kwargs(
        request
    )

    # --------------------------------------------------------
    # Request metadata
    # --------------------------------------------------------

    request_id = (
        "chatcmpl-"
        + uuid.uuid4().hex
    )

    started = time.time()

    print()
    print("=" * 70)
    print("INCOMING REQUEST")
    print("=" * 70)

    print(f"Request ID : {request_id}")
    print(f"Model      : {request.model or MODEL_ID}")
    print(
        f"Messages   : {len(request.messages)}"
    )
    print(
        f"Max tokens : {generation_kwargs['max_new_tokens']}"
    )
    print(
        f"Temperature: "
        f"{generation_kwargs.get('temperature', 'greedy')}"
    )
    print(
        f"Top-p      : "
        f"{generation_kwargs.get('top_p', 'N/A')}"
    )

    if request.reasoning_effort:

        print(
            f"Reasoning  : "
            f"{request.reasoning_effort}"
        )

    # --------------------------------------------------------
    # Generate
    # --------------------------------------------------------

    try:

        with generation_lock:

            output = pipe(
                messages,
                **generation_kwargs,
                return_full_text=False,
            )

    except torch.cuda.OutOfMemoryError:

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        raise HTTPException(
            status_code=507,
            detail=(
                "GPU out of memory during generation."
            ),
        )

    except Exception as e:

        print(
            f"Generation error: {type(e).__name__}: {e}"
        )

        raise HTTPException(
            status_code=500,
            detail=(
                f"Generation failed: {e}"
            ),
        )

    # --------------------------------------------------------
    # Extract response
    # --------------------------------------------------------

    text = extract_text(output)

    elapsed = time.time() - started

    # --------------------------------------------------------
    # Estimate token counts
    # --------------------------------------------------------

    completion_tokens = 0

    try:

        if hasattr(
            pipe,
            "tokenizer",
        ):

            completion_tokens = len(
                pipe.tokenizer.encode(
                    text,
                    add_special_tokens=False,
                )
            )

    except Exception:
        completion_tokens = 0

    # Prompt token count is difficult to estimate reliably
    # through the high-level any-to-any pipeline.
    prompt_tokens = 0

    total_tokens = (
        prompt_tokens
        + completion_tokens
    )

    # --------------------------------------------------------
    # Logging
    # --------------------------------------------------------

    print()
    print(
        f"Generation completed in "
        f"{elapsed:.2f}s"
    )

    print(
        f"Output tokens: "
        f"{completion_tokens}"
    )

    print(
        f"Output chars : "
        f"{len(text)}"
    )

    print("=" * 70)

    # --------------------------------------------------------
    # OpenAI-compatible response
    # --------------------------------------------------------

    return {
        "id": request_id,

        "object": "chat.completion",

        "created": int(time.time()),

        "model": MODEL_ID,

        "choices": [
            {
                "index": 0,

                "message": {
                    "role": "assistant",
                    "content": text,
                },

                "finish_reason": "stop",
            }
        ],

        "usage": {
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": total_tokens,
        },

        "x_inference_time_seconds": round(
            elapsed,
            3,
        ),
    }


# ============================================================
# 12. START UVICORN
# ============================================================

def run_server():

    uvicorn.run(
        app,
        host=HOST,
        port=PORT,
        log_level="info",
    )


# Avoid starting multiple servers if the cell
# is accidentally executed again.

if (
    "server_thread" not in globals()
    or not server_thread.is_alive()
):

    server_thread = threading.Thread(
        target=run_server,
        daemon=True,
    )

    server_thread.start()

    time.sleep(3)

else:

    print(
        "Existing API server is already running."
    )


print()
print("=" * 70)
print("GEMMA 4 E2B API READY")
print("=" * 70)

print(
    f"Local endpoint : "
    f"http://127.0.0.1:{PORT}"
)

print(
    f"OpenAI endpoint: "
    f"http://127.0.0.1:{PORT}/v1"
)

print(
    f"Model          : "
    f"{MODEL_ID}"
)

print(
    "API key        : configured"
)

print("=" * 70)

Generated a new GEMMA_API_KEY.
It is stored in the environment only.


INFO:     Started server process [57]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



GEMMA 4 E2B API READY
Local endpoint : http://127.0.0.1:8000
OpenAI endpoint: http://127.0.0.1:8000/v1
Model          : google/gemma-4-e2b-it
API key        : configured


In [2]:
# ============================================================
# CLOUDFLARE QUICK TUNNEL
# ============================================================

import os
import re
import time
import subprocess


# ------------------------------------------------------------
# 1. Install cloudflared
# ------------------------------------------------------------

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
    -O /usr/local/bin/cloudflared

!chmod +x /usr/local/bin/cloudflared


# ------------------------------------------------------------
# 2. Stop previous tunnel if it exists
# ------------------------------------------------------------

try:

    if (
        "cloudflared_process" in globals()
        and cloudflared_process is not None
    ):

        if cloudflared_process.poll() is None:

            print(
                "Stopping previous Cloudflare tunnel..."
            )

            cloudflared_process.terminate()

            try:
                cloudflared_process.wait(
                    timeout=5
                )
            except subprocess.TimeoutExpired:

                cloudflared_process.kill()

except Exception as e:

    print(
        f"Could not stop previous tunnel: {e}"
    )


# ------------------------------------------------------------
# 3. Start temporary public HTTPS tunnel
# ------------------------------------------------------------

print()
print("=" * 70)
print("STARTING CLOUDFLARE QUICK TUNNEL")
print("=" * 70)

print(
    f"Forwarding to: "
    f"http://127.0.0.1:{PORT}"
)

print()


cloudflared_process = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        f"http://127.0.0.1:{PORT}",
        "--no-autoupdate",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)


# ------------------------------------------------------------
# 4. Detect generated trycloudflare URL
# ------------------------------------------------------------

public_url = None

deadline = time.time() + 30


while time.time() < deadline:

    line = cloudflared_process.stdout.readline()

    if not line:

        if cloudflared_process.poll() is not None:
            break

        time.sleep(0.2)
        continue

    line = line.rstrip()

    print(line)

    match = re.search(
        r"https://[a-zA-Z0-9.-]+\.trycloudflare\.com",
        line,
    )

    if match:

        public_url = match.group(0)

        break


# ------------------------------------------------------------
# 5. Configure public API URL
# ------------------------------------------------------------

if public_url:

    PUBLIC_BASE_URL = (
        public_url
        + "/v1"
    )

    print()
    print("=" * 70)
    print("PUBLIC OPENAI-COMPATIBLE API")
    print("=" * 70)

    print()
    print("Base URL:")
    print(PUBLIC_BASE_URL)

    print()
    print("OpenAI SDK:")

    print(
        f'base_url="{PUBLIC_BASE_URL}"'
    )

    print()
    print("API key:")
    print("configured")

    print()
    print("Status:")
    print("Cloudflare tunnel is running.")

    print()
    print(
        "IMPORTANT:"
    )

    print(
        "- The trycloudflare URL is temporary."
    )

    print(
        "- It changes when the Kaggle session/tunnel restarts."
    )

    print(
        "- GEMMA_API_KEY is not printed."
    )

    print("=" * 70)


else:

    PUBLIC_BASE_URL = None

    print()
    print("=" * 70)
    print("CLOUDFLARE ERROR")
    print("=" * 70)

    print(
        "Could not automatically detect "
        "the Cloudflare URL."
    )

    print(
        "Check the cloudflared output above."
    )

    print("=" * 70)


STARTING CLOUDFLARE QUICK TUNNEL
Forwarding to: http://127.0.0.1:8000

2026-09-06T16:35:09Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-06T16:35:09Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-06T16:35:12Z INF +--------------------------------------------------------------------------------------------+
2026-09-06T16:35:12Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-0

In [8]:
# ============================================================
# PUBLIC API TEST
# ============================================================
assert PUBLIC_BASE_URL is not None

public_client = OpenAI(
    base_url=PUBLIC_BASE_URL,
    api_key=os.environ["GEMMA_API_KEY"],
)


response = public_client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {
            "role": "user",
            "content": (
                "Say hello and confirm that "
                "the public Gemma API is working."
            ),
        }
    ],
    max_tokens=100,
    temperature=0.2,
    top_p=0.95,
)


print(response.choices[0].message.content)

print()
print(
    "Public Base URL:",
    PUBLIC_BASE_URL,
)

print(
    "Inference time:",
    response.x_inference_time_seconds,
)


INCOMING REQUEST
Request ID : chatcmpl-8045fe9e2b7148ce9f4a70f3c9681c41
Model      : google/gemma-4-e2b-it
Messages   : 1
Max tokens : 100
Temperature: 0.2
Top-p      : 0.95
Generation error: NameError: name 'pipe' is not defined
INFO:     34.181.208.123:0 - "POST /v1/chat/completions HTTP/1.1" 500 Internal Server Error

INCOMING REQUEST
Request ID : chatcmpl-050e7618f753407fbefbed80c0eae836
Model      : google/gemma-4-e2b-it
Messages   : 1
Max tokens : 100
Temperature: 0.2
Top-p      : 0.95
Generation error: NameError: name 'pipe' is not defined
INFO:     34.181.208.123:0 - "POST /v1/chat/completions HTTP/1.1" 500 Internal Server Error

INCOMING REQUEST
Request ID : chatcmpl-334e1618fab548bb90f6de317e8f0f8f
Model      : google/gemma-4-e2b-it
Messages   : 1
Max tokens : 100
Temperature: 0.2
Top-p      : 0.95
Generation error: NameError: name 'pipe' is not defined
INFO:     34.181.208.123:0 - "POST /v1/chat/completions HTTP/1.1" 500 Internal Server Error


InternalServerError: Error code: 500 - {'detail': "Generation failed: name 'pipe' is not defined"}